# Week 5: Fetching Filings Through EdgarClient

Same logic as `reliable_fetch.py`, and the same task as Week 4's notebook — but now the client owns configuration, timeouts, and retries, instead of every call site managing its own `httpx.Client`. Requires `SEC_USER_AGENT` to be set in a `.env` file (see `.env.example`).

In [ ]:
import os

from dotenv import load_dotenv

from ai_finance_course.edgar import EdgarAPIError, EdgarClient, save_filings_json

TICKER = "AAPL"

load_dotenv()
user_agent = os.environ["SEC_USER_AGENT"]

try:
    with EdgarClient(user_agent=user_agent) as client:
        filings = client.get_filings_for_ticker(TICKER, limit=5)
except EdgarAPIError as exc:
    print(f"Could not fetch filings for {TICKER}: {exc}")
    raise

filings

## Save a Clean Subset

In [ ]:
save_filings_json(filings, "aapl_recent_filings.json")
print(f"Saved {len(filings)} filings for {TICKER}")

## Bonus: Paginating into Older Filings

`filings.recent` only covers roughly the most recent 1000 filings. Older history lives in separate paginated files, which `get_older_filings` walks through one page at a time.

In [ ]:
with EdgarClient(user_agent=user_agent) as client:
    cik = client.find_cik(TICKER)
    older_filings = client.get_older_filings(cik, page=0, limit=3)

older_filings